In [1]:
import numpy as np
from pathlib import Path

processed_dir = Path("../data/processed")
models_dir = Path("../models")

X_test = np.load(processed_dir / "X_test_scaled.npy")
y_test = np.load(processed_dir / "y_test.npy")

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nModels directory:")
print(models_dir.resolve())

print("\nFiles:")
for file in models_dir.iterdir():
    print(file.name)

X_test: (82332, 194)
y_test: (82332,)

Models directory:
C:\Users\hsv89\cyber_ml_project\models

Files:
pytorch_intrusion_detector.pth
xgboost_baseline.joblib


In [2]:
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Load saved XGBoost model
xgb_model = joblib.load(models_dir / "xgboost_baseline.joblib")

# Make predictions
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# Calculate metrics
xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred)
xgb_recall = recall_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_prob)

print("XGBoost model loaded successfully!")

print("\nAccuracy :", xgb_accuracy)
print("Precision:", xgb_precision)
print("Recall   :", xgb_recall)
print("F1 Score :", xgb_f1)
print("ROC-AUC  :", xgb_auc)

XGBoost model loaded successfully!

Accuracy : 0.8729534081523588
Precision: 0.8217805337171963
Recall   : 0.9822862437130504
F1 Score : 0.8948933861211037
ROC-AUC  : 0.9831144332146493


In [3]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class IntrusionDetector(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)


# Recreate model architecture
pytorch_model = IntrusionDetector(X_test.shape[1]).to(device)

# Load saved weights
model_path = models_dir / "pytorch_intrusion_detector.pth"

pytorch_model.load_state_dict(
    torch.load(model_path, map_location=device)
)

pytorch_model.eval()

print("PyTorch model loaded successfully!")
print("Device:", device)
print("Model file:", model_path.resolve())
print("Input features:", X_test.shape[1])

PyTorch model loaded successfully!
Device: cuda
Model file: C:\Users\hsv89\cyber_ml_project\models\pytorch_intrusion_detector.pth
Input features: 194


In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Convert test data to PyTorch tensor
X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
).to(device)

# Make predictions
pytorch_model.eval()

with torch.no_grad():
    logits = pytorch_model(X_test_tensor)
    probabilities = torch.sigmoid(logits)

    nn_prob = probabilities.cpu().numpy().ravel()
    nn_pred = (nn_prob >= 0.5).astype(int)

# Calculate metrics
nn_accuracy = accuracy_score(y_test, nn_pred)
nn_precision = precision_score(y_test, nn_pred)
nn_recall = recall_score(y_test, nn_pred)
nn_f1 = f1_score(y_test, nn_pred)
nn_auc = roc_auc_score(y_test, nn_prob)

print("PyTorch model evaluation:")
print("Accuracy :", nn_accuracy)
print("Precision:", nn_precision)
print("Recall   :", nn_recall)
print("F1 Score :", nn_f1)
print("ROC-AUC  :", nn_auc)

PyTorch model evaluation:
Accuracy : 0.8497789437885633
Precision: 0.7964495125723947
Recall   : 0.9768154945733698
F1 Score : 0.877459625483008
ROC-AUC  : 0.9741308552994008


In [5]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "XGBoost",
        "PyTorch Neural Network"
    ],
    "Accuracy": [
        xgb_accuracy,
        nn_accuracy
    ],
    "Precision": [
        xgb_precision,
        nn_precision
    ],
    "Recall": [
        xgb_recall,
        nn_recall
    ],
    "F1": [
        xgb_f1,
        nn_f1
    ],
    "ROC_AUC": [
        xgb_auc,
        nn_auc
    ]
})

comparison = comparison.round(4)

comparison

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,XGBoost,0.8730,0.8218,0.9823,0.8949,0.9831
1,PyTorch Neural Network,0.8498,0.7964,0.9768,0.8775,0.9741


In [6]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    
    predictions = (xgb_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df = threshold_df.round(4)

threshold_df

,Threshold,Accuracy,Precision,Recall,F1
0,0.30,0.8315,0.7675,0.9956,0.8668
1,0.35,0.8418,0.7797,0.9934,0.8736
2,0.40,0.8519,0.7925,0.9902,0.8804
3,0.45,0.8627,0.8070,0.9865,0.8878
4,0.50,0.8730,0.8218,0.9823,0.8949
5,0.55,0.8830,0.8376,0.9770,0.9020
6,0.60,0.8918,0.8536,0.9699,0.9080
7,0.65,0.9001,0.8694,0.9632,0.9139
8,0.70,0.9081,0.8868,0.9550,0.9196


In [7]:
from sklearn.model_selection import train_test_split

X_train_full = np.load(processed_dir / "X_train_scaled.npy")
y_train_full = np.load(processed_dir / "y_train.npy")

X_train_tune, X_val, y_train_tune, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)

print("Training subset:", X_train_tune.shape)
print("Validation set:", X_val.shape)

print("\nTraining labels:")
print("Normal:", (y_train_tune == 0).sum())
print("Attack:", (y_train_tune == 1).sum())

print("\nValidation labels:")
print("Normal:", (y_val == 0).sum())
print("Attack:", (y_val == 1).sum())

Training subset: (140272, 194)
Validation set: (35069, 194)

Training labels:
Normal: 44800
Attack: 95472

Validation labels:
Normal: 11200
Attack: 23869


In [8]:
from xgboost import XGBClassifier
import time

tune_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

start = time.time()

tune_model.fit(
    X_train_tune,
    y_train_tune
)

elapsed = time.time() - start

print("Tuning model trained successfully!")
print(f"Training time: {elapsed:.2f} seconds")

Tuning model trained successfully!
Training time: 2.12 seconds


In [9]:
val_prob = tune_model.predict_proba(X_val)[:, 1]

threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    val_pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, val_pred),
        "Precision": precision_score(y_val, val_pred),
        "Recall": recall_score(y_val, val_pred),
        "F1": f1_score(y_val, val_pred)
    })

val_threshold_df = pd.DataFrame(threshold_results).round(4)

val_threshold_df

,Threshold,Accuracy,Precision,Recall,F1
0,0.30,0.9511,0.9386,0.9931,0.9651
1,0.35,0.9548,0.9455,0.9907,0.9676
2,0.40,0.9578,0.9525,0.9872,0.9696
3,0.45,0.9589,0.9584,0.9823,0.9702
4,0.50,0.9594,0.9639,0.9768,0.9703
5,0.55,0.9589,0.9694,0.9703,0.9698
6,0.60,0.9575,0.9740,0.9634,0.9686
7,0.65,0.9547,0.9783,0.9547,0.9663
8,0.70,0.9513,0.9825,0.9453,0.9635
9,0.75,0.9448,0.9860,0.9321,0.9583


In [10]:
best_row = val_threshold_df.loc[val_threshold_df["F1"].idxmax()]

print("Best validation threshold:")
print(best_row)

Best validation threshold:
Threshold    0.5000
Accuracy     0.9594
Precision    0.9639
Recall       0.9768
F1           0.9703
Name: 4, dtype: float64


In [11]:
best_threshold = float(best_row["Threshold"])

test_prob = tune_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

final_accuracy = accuracy_score(y_test, test_pred)
final_precision = precision_score(y_test, test_pred)
final_recall = recall_score(y_test, test_pred)
final_f1 = f1_score(y_test, test_pred)
final_auc = roc_auc_score(y_test, test_prob)

print("Final test results")
print("Threshold :", best_threshold)
print("Accuracy  :", final_accuracy)
print("Precision :", final_precision)
print("Recall    :", final_recall)
print("F1 Score  :", final_f1)
print("ROC-AUC   :", final_auc)

Final test results
Threshold : 0.5
Accuracy  : 0.8747145702764417
Precision : 0.823124480944911
Recall    : 0.9838745257213447
F1 Score  : 0.8963493674447581
ROC-AUC   : 0.9841570148525831


In [12]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

Confusion Matrix:
[[27416  9584]
 [  731 44601]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.74      0.84     37000
           1       0.82      0.98      0.90     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.87     82332
weighted avg       0.89      0.87      0.87     82332



In [13]:
from xgboost import XGBClassifier
import time

final_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

start = time.time()

final_model.fit(X_train_full, y_train_full)

elapsed = time.time() - start

print("Final model trained successfully!")
print(f"Training time: {elapsed:.2f} seconds")

Final model trained successfully!
Training time: 2.44 seconds


In [14]:
final_prob = final_model.predict_proba(X_test)[:, 1]

final_threshold = 0.50
final_pred = (final_prob >= final_threshold).astype(int)

print("FINAL XGBOOST MODEL")
print("Threshold :", final_threshold)
print("Accuracy  :", accuracy_score(y_test, final_pred))
print("Precision :", precision_score(y_test, final_pred))
print("Recall    :", recall_score(y_test, final_pred))
print("F1 Score  :", f1_score(y_test, final_pred))
print("ROC-AUC   :", roc_auc_score(y_test, final_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_pred))

FINAL XGBOOST MODEL
Threshold : 0.5
Accuracy  : 0.8729534081523588
Precision : 0.8217805337171963
Recall    : 0.9822862437130504
F1 Score  : 0.8948933861211037
ROC-AUC   : 0.9831144332146493

Confusion Matrix:
[[27343  9657]
 [  803 44529]]


In [15]:
import joblib

final_model_path = models_dir / "final_xgboost_intrusion_detector.joblib"

joblib.dump(final_model, final_model_path)

print("Final model saved!")
print(final_model_path.resolve())
print("Exists:", final_model_path.exists())

Final model saved!
C:\Users\hsv89\cyber_ml_project\models\final_xgboost_intrusion_detector.joblib
Exists: True


In [16]:
import joblib

deployment_config = {
    "model_name": "final_xgboost_intrusion_detector",
    "threshold": 0.50,
    "input_features": X_test.shape[1],
    "positive_class": 1,
    "negative_class": 0
}

config_path = models_dir / "deployment_config.joblib"

joblib.dump(deployment_config, config_path)

print("Deployment configuration saved!")
print(config_path.resolve())
print(deployment_config)

Deployment configuration saved!
C:\Users\hsv89\cyber_ml_project\models\deployment_config.joblib
{'model_name': 'final_xgboost_intrusion_detector', 'threshold': 0.5, 'input_features': 194, 'positive_class': 1, 'negative_class': 0}
